In [1]:
import numpy as np
import pandas as pd

FILEPATH = r"/Users/riccardocornelli/Desktop/bitcoin/BTCUSD_1m_Binance.xlsx"

# ----------------------------
# LOAD (Excel) + BASIC CLEAN
# ----------------------------
df = pd.read_excel(FILEPATH)

# normalize column names (strip whitespace)
df.columns = [str(c).strip() for c in df.columns]

# Build a lowercase -> original mapping
col_map = {c.lower(): c for c in df.columns}

# Required columns (lowercase for lookup)
required_lower = ["open time", "open", "high", "low", "close", "volume"]
missing = [c for c in required_lower if c not in col_map]
if missing:
    raise ValueError(f"Missing columns in Excel: {missing}. Found: {list(df.columns)}")

# Rename to standardized names
rename_map = {
    col_map["open time"]: "Date",
    col_map["open"]:      "Open",
    col_map["high"]:      "High",
    col_map["low"]:       "Low",
    col_map["close"]:     "Close",
    col_map["volume"]:    "Volume",
}
df = df.rename(columns=rename_map)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").drop_duplicates(subset=["Date"], keep="last").reset_index(drop=True)

# ensure numeric
for c in ["Open", "High", "Low", "Close", "Volume"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["Open", "High", "Low", "Close", "Volume"]).reset_index(drop=True)

print(f"✅ Loaded rows: {len(df):,}")
print(f"✅ Columns: {list(df.columns)}")
df.head()

✅ Loaded rows: 999,999
✅ Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


,Date,Open,High,Low,Close,Volume
0,2017-08-17 04:00:00,4261.48,4261.48,4261.48,4261.48,1.775183
1,2017-08-17 04:01:00,4261.48,4261.48,4261.48,4261.48,0.000000
2,2017-08-17 04:02:00,4280.56,4280.56,4280.56,4280.56,0.261074
3,2017-08-17 04:03:00,4261.48,4261.48,4261.48,4261.48,0.012008
4,2017-08-17 04:04:00,4261.48,4261.48,4261.48,4261.48,0.140796


In [2]:
eps = 1e-12

# ----------------------------
# FEATURE ENGINEERING (STRICTLY CAUSAL)
# ----------------------------

# Returns / momentum (past-only)
df["ret_1m"]     = df["Close"].pct_change()
df["logret_1m"]  = np.log(df["Close"] + eps).diff()

for n in [5, 10, 20]:
    df[f"ret_{n}m"]          = df["Close"].pct_change(n)
    df[f"mom_{n}m"]          = df["Close"] / (df["Close"].shift(n) + eps) - 1.0
    df[f"vol_logret_{n}m"]   = df["logret_1m"].rolling(n, min_periods=n).std()

# Range / volatility proxies (past-only)
for n in [10, 20]:
    roll_high = df["High"].rolling(n, min_periods=n).max()
    roll_low  = df["Low"].rolling(n, min_periods=n).min()
    df[f"range_{n}m_norm"] = (roll_high - roll_low) / (df["Close"] + eps)

# ATR-like (True Range rolling mean) — strictly causal
prev_close = df["Close"].shift(1)
tr1 = (df["High"] - df["Low"]).abs()
tr2 = (df["High"] - prev_close).abs()
tr3 = (df["Low"]  - prev_close).abs()
df["true_range"] = np.maximum(tr1, np.maximum(tr2, tr3))
df["atr_14"] = df["true_range"].rolling(14, min_periods=14).mean()

# RSI-like (EWMA version, causal)
rsi_n = 14
delta = df["Close"].diff()
gain = delta.clip(lower=0.0)
loss = (-delta).clip(lower=0.0)
avg_gain = gain.ewm(alpha=1/rsi_n, adjust=False, min_periods=rsi_n).mean()
avg_loss = loss.ewm(alpha=1/rsi_n, adjust=False, min_periods=rsi_n).mean()
rs = avg_gain / (avg_loss + eps)
df["rsi_14"] = 100.0 - (100.0 / (1.0 + rs))

# MACD-like (causal EWMAs)
ema_fast = df["Close"].ewm(span=12, adjust=False, min_periods=12).mean()
ema_slow = df["Close"].ewm(span=26, adjust=False, min_periods=26).mean()
df["macd"] = ema_fast - ema_slow
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False, min_periods=9).mean()
df["macd_hist"] = df["macd"] - df["macd_signal"]

# Bollinger z-score (causal rolling)
bb_n = 20
bb_mid = df["Close"].rolling(bb_n, min_periods=bb_n).mean()
bb_std = df["Close"].rolling(bb_n, min_periods=bb_n).std()
df["bb_z_20"] = (df["Close"] - bb_mid) / (bb_std + eps)
df["bb_width_20"] = (2.0 * bb_std) / (bb_mid + eps)

# Distance to rolling max/min (support/resistance proxies, causal)
for n in [20, 50]:
    rmax = df["Close"].rolling(n, min_periods=n).max()
    rmin = df["Close"].rolling(n, min_periods=n).min()
    df[f"dist_to_max_{n}"] = df["Close"] / (rmax + eps) - 1.0
    df[f"dist_to_min_{n}"] = df["Close"] / (rmin + eps) - 1.0

# Volume features (causal)
df["vol_chg_1m"] = df["Volume"].pct_change()
df["vol_logchg_1m"] = np.log(df["Volume"] + eps).diff()
vol_mean_20 = df["Volume"].rolling(20, min_periods=20).mean()
vol_std_20  = df["Volume"].rolling(20, min_periods=20).std()
df["vol_z_20"] = (df["Volume"] - vol_mean_20) / (vol_std_20 + eps)

# Candle structure (causal, bar t only)
hl_range = (df["High"] - df["Low"]).abs()
body = df["Close"] - df["Open"]
upper_wick = df["High"] - np.maximum(df["Open"], df["Close"])
lower_wick = np.minimum(df["Open"], df["Close"]) - df["Low"]

df["candle_body_norm"] = body / (hl_range + eps)
df["upper_wick_norm"]  = upper_wick / (hl_range + eps)
df["lower_wick_norm"]  = lower_wick / (hl_range + eps)
df["close_loc"]        = (df["Close"] - df["Low"]) / (hl_range + eps)

# Simple trend proxies (causal)
df["ret_sign_1m"] = np.sign(df["ret_1m"].fillna(0.0))
df["trend_10m"] = df["ret_sign_1m"].rolling(10, min_periods=10).sum()
df["trend_20m"] = df["ret_sign_1m"].rolling(20, min_periods=20).sum()

print(f"✅ Base features added. Columns: {df.shape[1]}")

✅ Base features added. Columns: 41


In [3]:
# ============================
# EXTRA FEATURES (STRICTLY CAUSAL)
# ============================

# --- Time features (ex-ante, safe) ---
minute_of_day = df["Date"].dt.hour * 60 + df["Date"].dt.minute
df["tod_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
df["tod_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)

# --- Gap / microstructure-ish proxies (safe) ---
df["gap_open_prevclose"] = (df["Open"] / (df["Close"].shift(1) + eps)) - 1.0
df["hl_spread_norm"] = (df["High"] - df["Low"]) / (df["Close"] + eps)

# --- Rolling VWAP + distance to VWAP (safe) ---
tp = (df["High"] + df["Low"] + df["Close"]) / 3.0
pv = tp * df["Volume"]

for n in [20, 50]:
    vwap_n = pv.rolling(n, min_periods=n).sum() / (df["Volume"].rolling(n, min_periods=n).sum() + eps)
    df[f"vwap_{n}"] = vwap_n
    df[f"dist_to_vwap_{n}"] = (df["Close"] / (vwap_n + eps)) - 1.0

# --- Rolling position / percentile within window (safe) ---
for n in [20, 50]:
    rmax = df["Close"].rolling(n, min_periods=n).max()
    rmin = df["Close"].rolling(n, min_periods=n).min()
    df[f"close_pos_{n}"] = (df["Close"] - rmin) / ((rmax - rmin) + eps)

# --- OHLC-based volatility estimators (Parkinson, Garman-Klass) (safe) ---
log_hl = np.log((df["High"] + eps) / (df["Low"] + eps))
log_co = np.log((df["Close"] + eps) / (df["Open"] + eps))

parkinson_bar = (log_hl ** 2) / (4.0 * np.log(2.0))
gk_bar = 0.5 * (log_hl ** 2) - (2.0 * np.log(2.0) - 1.0) * (log_co ** 2)

for n in [10, 20]:
    df[f"vol_parkinson_{n}"] = np.sqrt(parkinson_bar.rolling(n, min_periods=n).mean() + eps)
    df[f"vol_gk_{n}"] = np.sqrt(gk_bar.rolling(n, min_periods=n).mean() + eps)

# --- EMA slopes (trend strength, safe) ---
for span in [10, 20, 50]:
    ema = df["Close"].ewm(span=span, adjust=False, min_periods=span).mean()
    df[f"ema_{span}"] = ema
    df[f"ema_slope_{span}"] = (ema - ema.shift(1)) / (df["Close"] + eps)

# --- Return distribution shape (skew/kurtosis, safe) ---
for n in [20, 50]:
    df[f"skew_logret_{n}"] = df["logret_1m"].rolling(n, min_periods=n).skew()
    df[f"kurt_logret_{n}"] = df["logret_1m"].rolling(n, min_periods=n).kurt()

# --- Autocorrelation proxies (safe) ---
r = df["logret_1m"]
for n in [20, 50]:
    df[f"autocorr1_{n}"] = r.rolling(n, min_periods=n).corr(r.shift(1))
    df[f"autocorr5_{n}"] = r.rolling(n, min_periods=n).corr(r.shift(5))

# --- Volatility of volatility (regime, safe) ---
for base in ["vol_logret_10m", "vol_logret_20m"]:
    if base in df.columns:
        df[f"vol_of_{base}_20"] = df[base].rolling(20, min_periods=20).std()

print(f"✅ Extra features added. Columns: {df.shape[1]}")

✅ Extra features added. Columns: 71


In [4]:
# ----------------------------
# FINAL DATAFRAME (drop initial NaNs from rolling/ewm windows)
# ----------------------------
df_feat = df.dropna().reset_index(drop=True)

print(f"✅ Features rows after dropna: {len(df_feat):,}")
print(f"✅ Total columns: {df_feat.shape[1]}")
df_feat.head()

✅ Features rows after dropna: 987,294
✅ Total columns: 71


,Date,Open,High,Low,Close,Volume,ret_1m,logret_1m,ret_5m,mom_5m,...,skew_logret_20,kurt_logret_20,skew_logret_50,kurt_logret_50,autocorr1_20,autocorr5_20,autocorr1_50,autocorr5_50,vol_of_vol_logret_10m_20,vol_of_vol_logret_20m_20
0,2017-08-17 04:55:00,4302.48,4313.60,4291.37,4291.37,4.635124,-0.005153,-0.005167,-0.004339,-0.004339,...,-0.914887,3.974932,0.855220,7.236059,-0.261717,-0.272988,-0.253354,-0.188393,0.000737,0.000149
1,2017-08-17 04:56:00,4313.60,4313.60,4313.60,4313.60,0.213562,0.005180,0.005167,-0.000005,-0.000005,...,-0.166836,2.606681,0.935887,5.392414,-0.509356,-0.210990,-0.385479,-0.154914,0.000772,0.000162
2,2017-08-17 04:57:00,4313.62,4313.62,4308.83,4308.83,1.332596,-0.001106,-0.001106,-0.001106,-0.001106,...,-0.090983,2.405766,0.950413,5.267902,-0.495291,-0.203678,-0.394079,-0.155831,0.000807,0.000174
3,2017-08-17 04:58:00,4308.83,4308.83,4308.83,4308.83,0.000000,0.000000,0.000000,-0.001106,-0.001106,...,-0.090983,2.405766,0.950413,5.267902,-0.490238,-0.210386,-0.390513,-0.177907,0.000839,0.000184
4,2017-08-17 04:59:00,4308.83,4308.83,4308.83,4308.83,0.560266,0.000000,0.000000,-0.001106,-0.001106,...,-0.090983,2.405766,0.950413,5.267902,-0.490238,-0.284970,-0.390513,-0.177907,0.000850,0.000191


In [5]:
from pathlib import Path

# Save on Desktop/bitcoin as: bitcoin_con_feature.csv
out_path = Path.home() / "Desktop" / "bitcoin" / "bitcoin_con_feature.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

df_feat.to_csv(out_path, index=False)
print(f"✅ Saved: {out_path}")

✅ Saved: /Users/riccardocornelli/Desktop/bitcoin/bitcoin_con_feature.csv


# 🎯 Target realistico: forward return a H minuti con soglia fee+slippage

**Logica:**
- `H = 20` minuti di orizzonte
- `X = 0.0012` (0.12%) soglia minima → copre ~0.08% fee round-trip + slippage
- **y = 1** → max forward return nei prossimi H minuti > +X (LONG profittevole)
- **y = 0** → min forward return nei prossimi H minuti < −X (SHORT profittevole / evitare)
- **y = NaN** → movimento troppo piccolo → **no trade** (escluso dal training)

In [6]:
# ============================
# TARGET: FORWARD RETURN CON SOGLIA FEE + SLIPPAGE
# ============================

H = 20        # orizzonte in minuti (barre)
X = 0.0012    # soglia minima (0.12%) per coprire fee + slippage

# Forward max/min return nei prossimi H minuti (ESCLUSA barra corrente → shift(-1))
# Per ogni barra t, guardiamo le barre [t+1, t+H]
fwd_max = df_feat["Close"].shift(-1).rolling(H, min_periods=1).max().shift(-H + 1)
fwd_min = df_feat["Close"].shift(-1).rolling(H, min_periods=1).min().shift(-H + 1)

# Forward return rispetto al Close corrente
fwd_ret_max = (fwd_max / df_feat["Close"]) - 1.0   # best case nei prossimi H min
fwd_ret_min = (fwd_min / df_feat["Close"]) - 1.0   # worst case nei prossimi H min

# Costruzione target
# y=1  se il max forward return > +X  (opportunità LONG)
# y=0  se il min forward return < -X  (opportunità SHORT / da evitare)
# y=NaN se nessuna delle due (no trade, escluso dal training)
#
# Se ENTRAMBE le condizioni sono vere (alta volatilità), diamo priorità
# alla direzione con il movimento più grande in valore assoluto.

cond_long  = fwd_ret_max > X
cond_short = fwd_ret_min < -X

target = pd.Series(np.nan, index=df_feat.index)

# Solo long
target[cond_long & ~cond_short] = 1.0
# Solo short
target[~cond_long & cond_short] = 0.0
# Entrambi → scegliamo la direzione dominante
both = cond_long & cond_short
target[both & (fwd_ret_max.abs() >= fwd_ret_min.abs())] = 1.0
target[both & (fwd_ret_max.abs() <  fwd_ret_min.abs())] = 0.0

df_feat["target"] = target
df_feat["fwd_ret_max"] = fwd_ret_max
df_feat["fwd_ret_min"] = fwd_ret_min

# Taglia le ultime H righe (non hanno futuro completo)
df_feat = df_feat.iloc[:-H].reset_index(drop=True)

# Statistiche
n_total   = len(df_feat)
n_labeled = df_feat["target"].notna().sum()
n_notrade = df_feat["target"].isna().sum()
n_long    = (df_feat["target"] == 1).sum()
n_short   = (df_feat["target"] == 0).sum()

print(f"{'='*50}")
print(f"📊 TARGET STATS  (H={H} min, soglia={X*100:.2f}%)")
print(f"{'='*50}")
print(f"  Righe totali:      {n_total:>10,}")
print(f"  Etichettate:       {n_labeled:>10,}  ({n_labeled/n_total*100:.1f}%)")
print(f"    ├─ LONG  (y=1):  {n_long:>10,}  ({n_long/n_labeled*100:.1f}% dei trade)")
print(f"    └─ SHORT (y=0):  {n_short:>10,}  ({n_short/n_labeled*100:.1f}% dei trade)")
print(f"  No-trade (NaN):    {n_notrade:>10,}  ({n_notrade/n_total*100:.1f}%)")
print(f"{'='*50}")

📊 TARGET STATS  (H=20 min, soglia=0.12%)
  Righe totali:         987,274
  Etichettate:          870,610  (88.2%)
    ├─ LONG  (y=1):     440,341  (50.6% dei trade)
    └─ SHORT (y=0):     430,269  (49.4% dei trade)
  No-trade (NaN):       116,664  (11.8%)


In [7]:
# ============================
# SALVA DATASET CON TARGET
# ============================

# Dataset completo (con NaN per no-trade)
out_full = Path.home() / "Desktop" / "bitcoin" / "bitcoin_features_target.csv"
df_feat.to_csv(out_full, index=False)
print(f"✅ Salvato dataset completo: {out_full}")
print(f"   Righe: {len(df_feat):,} | Colonne: {df_feat.shape[1]}")

# Dataset solo righe con etichetta (pronto per training)
df_train = df_feat.dropna(subset=["target"]).reset_index(drop=True)
df_train["target"] = df_train["target"].astype(int)

out_train = Path.home() / "Desktop" / "bitcoin" / "bitcoin_train_ready.csv"
df_train.to_csv(out_train, index=False)
print(f"✅ Salvato dataset training: {out_train}")
print(f"   Righe: {len(df_train):,} | Colonne: {df_train.shape[1]}")
print(f"   Distribuzione target: {df_train['target'].value_counts().to_dict()}")

✅ Salvato dataset completo: /Users/riccardocornelli/Desktop/bitcoin/bitcoin_features_target.csv
   Righe: 987,274 | Colonne: 74
✅ Salvato dataset training: /Users/riccardocornelli/Desktop/bitcoin/bitcoin_train_ready.csv
   Righe: 870,610 | Colonne: 74
   Distribuzione target: {1: 440341, 0: 430269}
✅ Salvato dataset training: /Users/riccardocornelli/Desktop/bitcoin/bitcoin_train_ready.csv
   Righe: 870,610 | Colonne: 74
   Distribuzione target: {1: 440341, 0: 430269}


# 🌲 Random Forest — Split Temporale + Soglia Operativa

**Regole fondamentali:**
1. **NO random split** → Train = passato, Val = blocco successivo, Test = ultime settimane
2. **class_weight="balanced"** → gestisce lo sbilanciamento creato dal filtro no-trade
3. **Soglia operativa alta** → NON tradare a p>0.5, ma cercare la soglia (es. 0.80) che massimizza la **precision nelle top probabilità** su validation
4. **Freeze** → la soglia scelta su val si applica "congelata" al test walk-forward

**Split temporale:**
- **Train**: primo ~70% cronologico
- **Validation**: ~15% successivo (per ottimizzare soglia)
- **Test**: ultimo ~15% (mai toccato fino alla valutazione finale)

In [8]:
# ============================
# SPLIT TEMPORALE (NO RANDOM!)
# ============================

# Lavoriamo SOLO sulle righe etichettate (no-trade già esclusi)
df_model = df_feat.dropna(subset=["target"]).reset_index(drop=True)
df_model["target"] = df_model["target"].astype(int)

# I dati sono GIÀ ordinati per Date (dal preprocessing)
# Split: 70% train | 15% validation | 15% test
n = len(df_model)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

df_tr  = df_model.iloc[:train_end].copy()
df_val = df_model.iloc[train_end:val_end].copy()
df_te  = df_model.iloc[val_end:].copy()

# Feature columns = tutto tranne Date, target, fwd_ret_max/min (sono info futura!)
drop_cols = ["Date", "target", "fwd_ret_max", "fwd_ret_min"]
feat_cols = [c for c in df_model.columns if c not in drop_cols]

# --- Pulizia: rimpiazza inf con NaN, poi NaN con 0 ---
for _df in [df_tr, df_val, df_te]:
    _df[feat_cols] = _df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

X_train, y_train = df_tr[feat_cols].values, df_tr["target"].values
X_val,   y_val   = df_val[feat_cols].values, df_val["target"].values
X_test,  y_test  = df_te[feat_cols].values, df_te["target"].values

# Verifica che non ci siano più inf/NaN
for name, arr in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    assert np.isfinite(arr).all(), f"❌ {name} contiene ancora inf/NaN!"

print(f"{'='*55}")
print(f"📅 SPLIT TEMPORALE (cronologico, NO shuffle)")
print(f"{'='*55}")
print(f"  Train:       {len(df_tr):>10,}  ({df_tr['Date'].min().date()} → {df_tr['Date'].max().date()})")
print(f"  Validation:  {len(df_val):>10,}  ({df_val['Date'].min().date()} → {df_val['Date'].max().date()})")
print(f"  Test:        {len(df_te):>10,}  ({df_te['Date'].min().date()} → {df_te['Date'].max().date()})")
print(f"  Features:    {len(feat_cols)}")
print(f"{'='*55}")
print(f"\n  Train  target dist: {pd.Series(y_train).value_counts().to_dict()}")
print(f"  Val    target dist: {pd.Series(y_val).value_counts().to_dict()}")
print(f"  Test   target dist: {pd.Series(y_test).value_counts().to_dict()}")
print(f"\n  ✅ Nessun inf/NaN nelle feature matrices")

📅 SPLIT TEMPORALE (cronologico, NO shuffle)
  Train:          609,427  (2017-08-17 → 2018-12-08)
  Validation:     130,591  (2018-12-08 → 2019-04-10)
  Test:           130,592  (2019-04-10 → 2019-07-16)
  Features:    70

  Train  target dist: {1: 307942, 0: 301485}
  Val    target dist: {1: 66124, 0: 64467}
  Test   target dist: {1: 66275, 0: 64317}

  ✅ Nessun inf/NaN nelle feature matrices


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report, precision_recall_curve,
    average_precision_score
)
import warnings, time
warnings.filterwarnings("ignore", category=UserWarning)

# ============================
# TRAIN RANDOM FOREST
# ============================

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=200,       # anti-overfit su dati 1-min
    max_features="sqrt",
    class_weight="balanced",    # ← gestisce sbilanciamento
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

print("⏳ Training Random Forest (class_weight='balanced') ...")
t0 = time.time()
rf.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"✅ Fit completato in {elapsed:.1f}s")

# Probabilità classe 1 (LONG)
prob_val  = rf.predict_proba(X_val)[:, 1]
prob_test = rf.predict_proba(X_test)[:, 1]

# AUC di base (con soglia 0.5 standard)
auc_val  = roc_auc_score(y_val, prob_val)
auc_test = roc_auc_score(y_test, prob_test)
ap_val   = average_precision_score(y_val, prob_val)
ap_test  = average_precision_score(y_test, prob_test)

print(f"\n{'='*55}")
print(f"📈 METRICHE BASE (soglia default 0.5)")
print(f"{'='*55}")
print(f"  Validation AUC-ROC:  {auc_val:.4f}   |  Avg Precision: {ap_val:.4f}")
print(f"  Test       AUC-ROC:  {auc_test:.4f}   |  Avg Precision: {ap_test:.4f}")
print(f"{'='*55}")

# Top-10 feature importance
importances = pd.Series(rf.feature_importances_, index=feat_cols).sort_values(ascending=False)
print(f"\n🏆 Top 15 Feature Importances:")
for i, (feat, imp) in enumerate(importances.head(15).items(), 1):
    print(f"  {i:2d}. {feat:<30s}  {imp:.4f}")

⏳ Training Random Forest (class_weight='balanced') ...
✅ Fit completato in 178.1s
✅ Fit completato in 178.1s

📈 METRICHE BASE (soglia default 0.5)
  Validation AUC-ROC:  0.5320   |  Avg Precision: 0.5331
  Test       AUC-ROC:  0.5422   |  Avg Precision: 0.5428

🏆 Top 15 Feature Importances:
   1. Volume                          0.0394
   2. ema_slope_50                    0.0364
   3. close_loc                       0.0330
   4. ema_slope_10                    0.0326
   5. close_pos_20                    0.0303
   6. ema_slope_20                    0.0253
   7. dist_to_max_20                  0.0251
   8. dist_to_max_50                  0.0243
   9. close_pos_50                    0.0216
  10. hl_spread_norm                  0.0200
  11. ret_1m                          0.0199
  12. dist_to_min_20                  0.0198
  13. dist_to_min_50                  0.0196
  14. macd_signal                     0.0193
  15. logret_1m                       0.0193

📈 METRICHE BASE (soglia default 

In [10]:
# ============================
# OTTIMIZZAZIONE SOGLIA SU VALIDATION (poi freeze)
# ============================
# Obiettivo: trovare la soglia P che massimizza la precision LONG
# mantenendo un numero minimo di trade (almeno 1% delle barre di val).

min_trades_frac = 0.01  # almeno 1% delle barre devono essere "trade"
thresholds_to_try = np.arange(0.50, 0.96, 0.01)

results = []
for thr in thresholds_to_try:
    # LONG: p >= thr  |  SHORT: p <= (1-thr)
    pred_long  = prob_val >= thr
    pred_short = prob_val <= (1 - thr)
    pred_trade = pred_long | pred_short

    n_trades = pred_trade.sum()
    if n_trades < min_trades_frac * len(y_val):
        continue

    # Precision LONG: tra quelli predetti LONG, quanti sono davvero y=1?
    n_l = pred_long.sum()
    if n_l > 0:
        prec_long = float((y_val[pred_long] == 1).mean())
    else:
        prec_long = float("nan")

    # Precision SHORT: tra quelli predetti SHORT, quanti sono davvero y=0?
    n_s = pred_short.sum()
    if n_s > 0:
        prec_short = float((y_val[pred_short] == 0).mean())
    else:
        prec_short = float("nan")

    # Precision media ponderata
    _pl = prec_long  if not np.isnan(prec_long)  else 0.0
    _ps = prec_short if not np.isnan(prec_short) else 0.0
    if n_l + n_s > 0:
        prec_avg = (_pl * n_l + _ps * n_s) / (n_l + n_s)
    else:
        prec_avg = 0.0

    results.append({
        "threshold": thr,
        "n_trades": int(n_trades),
        "n_long": int(n_l),
        "n_short": int(n_s),
        "prec_long": prec_long,
        "prec_short": prec_short,
        "prec_avg": prec_avg,
        "trade_frac": n_trades / len(y_val),
    })

df_thr = pd.DataFrame(results)

# Soglia ottimale: massima precision media con almeno un po' di trade
best_row = df_thr.loc[df_thr["prec_avg"].idxmax()]
BEST_THR = best_row["threshold"]

print(f"{'='*65}")
print(f"🔍 SWEEP SOGLIA SU VALIDATION")
print(f"{'='*65}")
print(df_thr.to_string(index=False, float_format="{:.4f}".format))
print(f"{'='*65}")
print(f"\n🏆 SOGLIA OTTIMALE (validation): {BEST_THR:.2f}")
print(f"   Precision LONG:  {best_row['prec_long']:.4f}  ({best_row['n_long']:.0f} trade)")
print(f"   Precision SHORT: {best_row['prec_short']:.4f}  ({best_row['n_short']:.0f} trade)")
print(f"   Precision AVG:   {best_row['prec_avg']:.4f}")
print(f"   Trade fraction:  {best_row['trade_frac']:.2%}")
print(f"\n⚡ Questa soglia viene ora CONGELATA e applicata al test set.")

🔍 SWEEP SOGLIA SU VALIDATION
 threshold  n_trades  n_long  n_short  prec_long  prec_short  prec_avg  trade_frac
    0.5000    130591   39786    90805     0.5323      0.5050    0.5134      1.0000
    0.5100     99210   25290    73920     0.5421      0.5087    0.5172      0.7597
    0.5200     70963   14892    56071     0.5541      0.5182    0.5257      0.5434
    0.5300     47868    8128    39740     0.5742      0.5312    0.5385      0.3665
    0.5400     30514    4138    26376     0.5962      0.5443    0.5513      0.2337
    0.5500     18789    1893    16896     0.5990      0.5626    0.5662      0.1439
    0.5600     11730     786    10944     0.6196      0.5834    0.5858      0.0898
    0.5700      7765     306     7459     0.6242      0.5931    0.5943      0.0595
    0.5800      5195     125     5070     0.6480      0.5957    0.5969      0.0398
    0.5900      3233      68     3165     0.6324      0.6025    0.6032      0.0248
    0.6000      1703      38     1665     0.6316      0.61

In [11]:
# ============================
# VALUTAZIONE FINALE SU TEST (soglia congelata da validation)
# ============================

# Applica soglia congelata
test_pred_long  = prob_test >= BEST_THR
test_pred_short = prob_test <= (1 - BEST_THR)
test_pred_trade = test_pred_long | test_pred_short
test_no_trade   = ~test_pred_trade

# Metriche LONG
if test_pred_long.sum() > 0:
    prec_long_test = (y_test[test_pred_long] == 1).mean()
    # Return medio reale dei trade LONG predetti
    ret_long = df_te.loc[test_pred_long, "fwd_ret_max"].mean()
else:
    prec_long_test, ret_long = np.nan, np.nan

# Metriche SHORT
if test_pred_short.sum() > 0:
    prec_short_test = (y_test[test_pred_short] == 0).mean()
    ret_short = df_te.loc[test_pred_short, "fwd_ret_min"].mean()
else:
    prec_short_test, ret_short = np.nan, np.nan

n_long_test  = test_pred_long.sum()
n_short_test = test_pred_short.sum()
n_trade_test = test_pred_trade.sum()
n_skip_test  = test_no_trade.sum()

print(f"{'='*65}")
print(f"🧪 TEST FINALE  (soglia congelata = {BEST_THR:.2f})")
print(f"{'='*65}")
print(f"  AUC-ROC (tutte le barre):     {auc_test:.4f}")
print(f"  Avg Precision (tutte):         {ap_test:.4f}")
print(f"{'─'*65}")
print(f"  Barre totali test:             {len(y_test):,}")
print(f"  Trade eseguiti:                {n_trade_test:,}  ({n_trade_test/len(y_test):.1%})")
print(f"    ├─ LONG  predetti:           {n_long_test:,}")
print(f"    │   └─ Precision LONG:       {prec_long_test:.4f}")
print(f"    └─ SHORT predetti:           {n_short_test:,}")
print(f"        └─ Precision SHORT:      {prec_short_test:.4f}")
print(f"  No-trade (skip):               {n_skip_test:,}  ({n_skip_test/len(y_test):.1%})")
print(f"{'='*65}")

# --- Analisi per decili di probabilità (top-confidence) ---
print(f"\n📊 PRECISION PER DECILI DI CONFIDENZA (TEST)")
print(f"{'─'*65}")
print(f"  {'Decile':<12} {'Range prob':<18} {'N barre':>8} {'%y=1':>8} {'Prec':>8}")
print(f"{'─'*65}")

deciles = np.linspace(0, 1, 11)
for i in range(len(deciles) - 1):
    lo, hi = deciles[i], deciles[i + 1]
    mask = (prob_test >= lo) & (prob_test < hi + 1e-9)
    n_in = mask.sum()
    if n_in > 0:
        pct_1 = (y_test[mask] == 1).mean()
        label = f"D{i+1}"
        print(f"  {label:<12} [{lo:.2f}, {hi:.2f})  {n_in:>8,}  {pct_1:>7.1%}  {'✅' if pct_1 > 0.55 else '❌' if pct_1 < 0.45 else '➖'}")

print(f"{'─'*65}")
print(f"\n💡 Operativamente: trada SOLO quando p >= {BEST_THR:.2f} (LONG)")
print(f"   o p <= {1-BEST_THR:.2f} (SHORT). Tutto il resto → SKIP.")

🧪 TEST FINALE  (soglia congelata = 0.60)
  AUC-ROC (tutte le barre):     0.5422
  Avg Precision (tutte):         0.5428
─────────────────────────────────────────────────────────────────
  Barre totali test:             130,592
  Trade eseguiti:                4,719  (3.6%)
    ├─ LONG  predetti:           451
    │   └─ Precision LONG:       0.6341
    └─ SHORT predetti:           4,268
        └─ Precision SHORT:      0.5872
  No-trade (skip):               125,873  (96.4%)

📊 PRECISION PER DECILI DI CONFIDENZA (TEST)
─────────────────────────────────────────────────────────────────
  Decile       Range prob          N barre     %y=1     Prec
─────────────────────────────────────────────────────────────────
  D4           [0.30, 0.40)     4,268    41.3%  ❌
  D5           [0.40, 0.50)    67,691    48.5%  ➖
  D6           [0.50, 0.60)    58,182    54.0%  ➖
  D7           [0.60, 0.70)       451    63.4%  ✅
─────────────────────────────────────────────────────────────────

💡 Operativament

In [12]:
# ============================
# PERFORMANCE METRICS (WEEKLY)
# Returns, Volatility, Sharpe (NOT annualized)
# Weekly gain/loss table + summary
# ============================
# Builds equity curve from test-set predictions (threshold frozen from validation).

import numpy as np
import pandas as pd

# ---- Strategy parameters ----
FEE_ONE_WAY = 0.0006  # 0.06% per lato → 0.12% round-trip (fee + slippage)

# ---- Determine position at each bar ----
# +1 = LONG, -1 = SHORT, 0 = FLAT (skip)
position = np.zeros(len(df_te), dtype=float)
position[prob_test >= BEST_THR]       =  1.0   # LONG
position[prob_test <= (1 - BEST_THR)] = -1.0   # SHORT

# 1-bar forward return (return from Close[t] to Close[t+1])
close_arr = df_te["Close"].values.astype(float)
fwd_ret = np.zeros(len(close_arr))
fwd_ret[:-1] = (close_arr[1:] / close_arr[:-1]) - 1.0

# ---- Gross strategy return (before fees) ----
# position[t] * fwd_ret[t] = P&L from holding position[t] over bar t→t+1
gross_ret = position * fwd_ret

# ---- Fee logic: pay fee only when position CHANGES ----
# Change in position → abs(Δpos) * fee_one_way
# E.g. flat→long = |+1-0|=1 → 1 × 0.06% = 0.06%
#      long→short = |-1-1|=2 → 2 × 0.06% = 0.12%
#      long→flat  = |0-1|=1  → 1 × 0.06% = 0.06%
pos_change = np.abs(np.diff(position, prepend=0.0))  # prepend 0 = start flat
fee_cost = pos_change * FEE_ONE_WAY

# ---- Net strategy return ----
net_ret = gross_ret - fee_cost

# Equity curve (compounded)
equity = (1.0 + net_ret).cumprod()

# Build a time-indexed Series
dates_idx = pd.to_datetime(df_te["Date"].values)
equity_ts = pd.Series(equity, index=dates_idx, name="equity")
equity_ts = equity_ts.sort_index()

# ---- Weekly resampling ----
weekly_equity = equity_ts.resample("W-SUN").last().dropna()
weekly_ret = weekly_equity.pct_change().dropna()

# ----------------------------
# METRICS
# ----------------------------
total_return = (weekly_equity.iloc[-1] / weekly_equity.iloc[0]) - 1.0
vol_weekly = weekly_ret.std(ddof=1)
mean_weekly = weekly_ret.mean()
sharpe_weekly = np.nan if vol_weekly == 0 else (mean_weekly / vol_weekly)

# Weekly gain/loss table
weekly_tbl = pd.DataFrame({
    "WeekEnd": weekly_ret.index,
    "WeeklyReturn": weekly_ret.values,
    "WeeklyPnL_EquityDelta": (weekly_equity.diff().reindex(weekly_ret.index)).values
}).set_index("WeekEnd")

wins = weekly_tbl[weekly_tbl["WeeklyReturn"] > 0]
losses = weekly_tbl[weekly_tbl["WeeklyReturn"] < 0]

summary = {
    "Weeks_total": int(len(weekly_tbl)),
    "Weeks_win": int(len(wins)),
    "Weeks_loss": int(len(losses)),
    "Win_rate": float(len(wins) / len(weekly_tbl)) if len(weekly_tbl) else np.nan,
    "Avg_win_week_return": float(wins["WeeklyReturn"].mean()) if len(wins) else np.nan,
    "Avg_loss_week_return": float(losses["WeeklyReturn"].mean()) if len(losses) else np.nan,
    "Total_return": float(total_return),
    "Vol_weekly": float(vol_weekly),
    "Sharpe_weekly_NOT_annualized": float(sharpe_weekly),
}

best_week = weekly_tbl["WeeklyReturn"].idxmax()
worst_week = weekly_tbl["WeeklyReturn"].idxmin()

# Trade stats
n_long_bars  = int((position ==  1).sum())
n_short_bars = int((position == -1).sum())
n_flat_bars  = int((position ==  0).sum())
n_entries    = int((pos_change > 0).sum())  # number of position changes (entries/exits)
total_fee_paid = float(fee_cost.sum() * 100)

print(f"{'='*60}")
print(f"📈 PERFORMANCE (WEEKLY, NOT annualized)")
print(f"   Soglia congelata: {BEST_THR:.2f} | Fee one-way: {FEE_ONE_WAY*100:.2f}%")
print(f"{'='*60}")
print(f"  Barre LONG:  {n_long_bars:>8,}  |  Barre SHORT: {n_short_bars:>8,}  |  FLAT: {n_flat_bars:>8,}")
print(f"  Cambi posizione (entries/exits): {n_entries:,}")
print(f"  Fee totali pagate: {total_fee_paid:.2f}% dell'equity")
print(f"{'─'*60}")
print(f"  Total Return:      {summary['Total_return']:.4%}")
print(f"  Weekly Volatility: {summary['Vol_weekly']:.4%}")
print(f"  Weekly Sharpe:     {summary['Sharpe_weekly_NOT_annualized']:.3f}")
print()
print(f"  Weeks: {summary['Weeks_total']} | Win: {summary['Weeks_win']} | Loss: {summary['Weeks_loss']} | Win rate: {summary['Win_rate']:.2%}")
print(f"  Avg Win Week:  {summary['Avg_win_week_return']:.4%}" if len(wins) else "  Avg Win Week:  N/A")
print(f"  Avg Loss Week: {summary['Avg_loss_week_return']:.4%}" if len(losses) else "  Avg Loss Week: N/A")
print()
print(f"  Best Week End:  {best_week.date()} | Return: {weekly_tbl.loc[best_week,'WeeklyReturn']:.4%} | ΔEquity: {weekly_tbl.loc[best_week,'WeeklyPnL_EquityDelta']:.6f}")
print(f"  Worst Week End: {worst_week.date()} | Return: {weekly_tbl.loc[worst_week,'WeeklyReturn']:.4%} | ΔEquity: {weekly_tbl.loc[worst_week,'WeeklyPnL_EquityDelta']:.6f}")

# Show the weekly table (sorted by time)
weekly_tbl_display = weekly_tbl.copy()
weekly_tbl_display["WeeklyReturn_pct"] = (weekly_tbl_display["WeeklyReturn"] * 100).round(3)
weekly_tbl_display = weekly_tbl_display.drop(columns=["WeeklyReturn"])
print(f"\n{'='*60}")
print("📋 WEEKLY TABLE (first 10 / last 10)")
print(f"{'='*60}")
weekly_tbl_display.head(10), weekly_tbl_display.tail(10)

📈 PERFORMANCE (WEEKLY, NOT annualized)
   Soglia congelata: 0.60 | Fee one-way: 0.06%
  Barre LONG:       451  |  Barre SHORT:    4,268  |  FLAT:  125,873
  Cambi posizione (entries/exits): 4,013
  Fee totali pagate: 240.84% dell'equity
────────────────────────────────────────────────────────────
  Total Return:      -84.7546%
  Weekly Volatility: 10.9005%
  Weekly Sharpe:     -1.092

  Weeks: 14 | Win: 1 | Loss: 13 | Win rate: 7.14%
  Avg Win Week:  0.1101%
  Avg Loss Week: -12.8306%

  Best Week End:  2019-07-21 | Return: 0.1101% | ΔEquity: 0.000153
  Worst Week End: 2019-05-12 | Return: -33.2408% | ΔEquity: -0.222254

📋 WEEKLY TABLE (first 10 / last 10)


(            WeeklyPnL_EquityDelta  WeeklyReturn_pct
 WeekEnd                                            
 2019-04-21              -0.042837            -4.693
 2019-04-28              -0.083657            -9.616
 2019-05-05              -0.117675           -14.966
 2019-05-12              -0.222254           -33.241
 2019-05-19              -0.132742           -29.739
 2019-05-26              -0.067920           -21.657
 2019-06-02              -0.029525           -12.017
 2019-06-09              -0.035427           -16.388
 2019-06-16              -0.029651           -16.404
 2019-06-23              -0.010642            -7.043,
             WeeklyPnL_EquityDelta  WeeklyReturn_pct
 WeekEnd                                            
 2019-05-19              -0.132742           -29.739
 2019-05-26              -0.067920           -21.657
 2019-06-02              -0.029525           -12.017
 2019-06-09              -0.035427           -16.388
 2019-06-16              -0.029651           

In [13]:

# ==============================================================================
# STAGE 4 — SOLUZIONE REALE BASATA SULLA LETTERATURA
# ==============================================================================
# Problema identificato: AUC ~0.53 a 1-min NON copre i costi di transazione.
#
# SOLUZIONI (da letteratura e practitioner):
# 1. RICAMPIONAMENTO a 5 min  → riduce trade × 5x, migliora SNR
# 2. TARGET CORRETTO          → close-to-close netto (no lookahead max/min)
# 3. META-LABELING            → filtro secondario su alta-confidenza
# 4. REGIME FILTER            → no trade in high-vol regime (bear market 2022)
# 5. SOGLIA ADATTIVA          → calibrata su precision netta (post-fee)
# ==============================================================================

import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Parametri ──────────────────────────────────────────────────────────────
RESAMPLE_MIN    = 5        # aggrega 1-min in 5-min bars
H_BARS          = 6        # orizzonte previsione: 6 barre × 5min = 30 min
FEE_RT          = 0.0010   # 0.10% round-trip (fee + slippage, Binance taker ~0.075%)
MIN_EDGE        = FEE_RT * 1.5  # il modello deve prevedere almeno 1.5× il costo
VOL_REGIME_MULT = 2.0      # soglia regime: vol > 2× mediana → no trade
ANNUAL_VOL_TGT  = 0.15     # target 15% annuo
BARS_PER_YEAR   = 252 * 24 * 60 // RESAMPLE_MIN   # barre 5-min per anno

print("=" * 70)
print("  STAGE 4 — Soluzione: Resample 5m + MetaLabel + Regime Filter")
print("=" * 70)

# ─────────────────────────────────────────────────────────────────────────────
# 1. CARICA DATI E RICAMPIONA A 5 MINUTI
# ─────────────────────────────────────────────────────────────────────────────
raw_path = Path.home() / "Desktop" / "bitcoin" / "BTCUSD_1m_Binance.xlsx"
print(f"\n[1/6] Caricamento dati da: {raw_path.name}")
raw = pd.read_excel(raw_path)
raw.columns = [str(c).strip() for c in raw.columns]
col_map = {c.lower(): c for c in raw.columns}
raw = raw.rename(columns={
    col_map["open time"]: "Date",
    col_map["open"]:  "Open",
    col_map["high"]:  "High",
    col_map["low"]:   "Low",
    col_map["close"]: "Close",
    col_map["volume"]:"Volume",
})
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
raw = (raw.dropna(subset=["Date"])
         .sort_values("Date")
         .drop_duplicates(subset=["Date"], keep="last")
         .reset_index(drop=True))
for c in ["Open","High","Low","Close","Volume"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw = raw.dropna().set_index("Date")

# Resample OHLCV
r = f"{RESAMPLE_MIN}min"
d5 = raw.resample(r).agg(
    Open=("Open", "first"),
    High=("High", "max"),
    Low=("Low", "min"),
    Close=("Close", "last"),
    Volume=("Volume", "sum"),
).dropna().reset_index()
print(f"   1-min bars: {len(raw):,}  →  5-min bars: {len(d5):,}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING SU BARRE 5-MINUTI
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[2/6] Feature engineering su barre {RESAMPLE_MIN}-min ...")
eps = 1e-12
df = d5.copy()

# Log-return
df["logret"] = np.log(df["Close"] + eps).diff()

# Momentum multi-lag
for n in [2, 3, 5, 10, 20, 40]:
    df[f"mom_{n}"] = df["Close"].pct_change(n)

# Volatilità rolling (stima sigma per bar)
for n in [10, 20, 40, 80]:
    df[f"vol_{n}"] = df["logret"].rolling(n, min_periods=n).std()

# RSI
rsi_n = 14
delta = df["Close"].diff()
avg_g = delta.clip(lower=0).ewm(alpha=1/rsi_n, adjust=False).mean()
avg_l = (-delta).clip(lower=0).ewm(alpha=1/rsi_n, adjust=False).mean()
df["rsi"] = 100 - 100 / (1 + avg_g / (avg_l + eps))

# MACD
df["macd"] = df["Close"].ewm(span=12, adjust=False).mean() - df["Close"].ewm(span=26, adjust=False).mean()
df["macd_sig"] = df["macd"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd"] - df["macd_sig"]

# Bollinger z-score
for n in [20, 40]:
    mid = df["Close"].rolling(n).mean()
    std = df["Close"].rolling(n).std()
    df[f"bb_z_{n}"] = (df["Close"] - mid) / (std + eps)

# ATR normalizzato
tr = pd.concat([
    (df["High"] - df["Low"]).abs(),
    (df["High"] - df["Close"].shift()).abs(),
    (df["Low"]  - df["Close"].shift()).abs()
], axis=1).max(axis=1)
df["atr_14_norm"] = tr.ewm(span=14, adjust=False).mean() / (df["Close"] + eps)

# Struttura candle
hl = (df["High"] - df["Low"]).abs()
df["body_norm"] = (df["Close"] - df["Open"]) / (hl + eps)
df["close_loc"] = (df["Close"] - df["Low"]) / (hl + eps)

# Volume z-score
df["vol_z"] = (df["Volume"] - df["Volume"].rolling(20).mean()) / (df["Volume"].rolling(20).std() + eps)

# EMA slope
for s in [10, 20, 50]:
    ema = df["Close"].ewm(span=s, adjust=False).mean()
    df[f"ema_sl_{s}"] = (ema - ema.shift(1)) / (df["Close"] + eps)

# Distanza da massimo/minimo rolling
for n in [20, 50]:
    rmax = df["Close"].rolling(n).max()
    rmin = df["Close"].rolling(n).min()
    df[f"dist_max_{n}"] = df["Close"] / (rmax + eps) - 1
    df[f"dist_min_{n}"] = df["Close"] / (rmin + eps) - 1

# Time of day
mod = df["Date"].dt.hour * 60 + df["Date"].dt.minute
df["tod_sin"] = np.sin(2 * np.pi * mod / 1440)
df["tod_cos"] = np.cos(2 * np.pi * mod / 1440)

# Autocorrelazione
df["autocorr1"] = df["logret"].rolling(20).corr(df["logret"].shift(1))

# ─────────────────────────────────────────────────────────────────────────────
# 3. TARGET CORRETTO: close-to-close a H barre (NO lookahead max/min)
# ─────────────────────────────────────────────────────────────────────────────
# Segnale binario: 1 se return netto futuro > fee, 0 altrimenti
fwd_ret_h = df["Close"].shift(-H_BARS) / df["Close"] - 1.0
# y=1 → LONG profittevole (return > fee RT)
# y=0 → SHORT profittevole (return < -fee RT)
# y=NaN → zona "morta" → esclusa
target = pd.Series(np.nan, index=df.index)
target[fwd_ret_h >  FEE_RT] = 1.0
target[fwd_ret_h < -FEE_RT] = 0.0
df["target"] = target
df["fwd_ret_h"] = fwd_ret_h

# REGIME FILTER: volatilità rolling 40 barre
df["vol_regime"] = df["logret"].rolling(40, min_periods=40).std()
vol_median = df["vol_regime"].median()
df["low_vol_regime"] = (df["vol_regime"] < VOL_REGIME_MULT * vol_median).astype(float)

df = df.dropna().reset_index(drop=True)
print(f"   Barre totali: {len(df):,}")
n_lab = df["target"].notna().sum()
n_lv = df["low_vol_regime"].sum()
print(f"   Barre etichettate: {n_lab:,} ({n_lab/len(df):.1%})")
print(f"   Barre low-vol regime: {int(n_lv):,} ({n_lv/len(df):.1%})")

# ─────────────────────────────────────────────────────────────────────────────
# 4. SPLIT TEMPORALE
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[3/6] Split temporale (70/15/15) ...")

df_model = df.dropna(subset=["target"]).reset_index(drop=True)
df_model["target"] = df_model["target"].astype(int)

n = len(df_model)
i_tr  = int(n * 0.70)
i_val = int(n * 0.85)

df_tr  = df_model.iloc[:i_tr].copy()
df_val = df_model.iloc[i_tr:i_val].copy()
df_te  = df_model.iloc[i_val:].copy()

drop_cols = ["Date", "target", "fwd_ret_h", "low_vol_regime", "vol_regime",
             "Open", "High", "Low", "Close", "Volume", "logret"]
feat_cols = [c for c in df_model.columns if c not in drop_cols]

for _d in [df_tr, df_val, df_te]:
    _d[feat_cols] = _d[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

X_tr, y_tr   = df_tr[feat_cols].values,  df_tr["target"].values
X_val, y_val = df_val[feat_cols].values, df_val["target"].values
X_te, y_te   = df_te[feat_cols].values,  df_te["target"].values

scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_te)

print(f"   Train:  {len(df_tr):,} | Val: {len(df_val):,} | Test: {len(df_te):,}")
print(f"   Feature: {len(feat_cols)}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. MODELLO PRIMARIO + META-LABELING
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[4/6] Training modelli (RF primario + Meta-label LR) ...")

# --- Modello Primario: RF ---
rf_primary = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=100,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_primary.fit(X_tr_s, y_tr)
p_primary_val = rf_primary.predict_proba(X_val_s)[:, 1]
p_primary_te  = rf_primary.predict_proba(X_te_s)[:, 1]

auc_primary_val = roc_auc_score(y_val, p_primary_val)
auc_primary_te  = roc_auc_score(y_te,  p_primary_te)
print(f"   RF Primario — AUC val: {auc_primary_val:.4f}  |  AUC test: {auc_primary_te:.4f}")

# --- Meta-Labeling (Lopez de Prado 2018) ---
# Il meta-labeler risponde a: "il modello primario ha ragione qui?"
# Costruiamo il dataset per il meta-labeler:
#   - Features = features originali + p_primary + |p_primary - 0.5| (confidenza)
#   - Target meta = 1 se il modello primario ha ragione, 0 altrimenti

# Previsione del modello primario su train (con out-of-fold per evitare leakage)
from sklearn.model_selection import cross_val_predict
p_primary_train_oof = cross_val_predict(
    RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=100,
                           max_features="sqrt", class_weight="balanced",
                           random_state=42, n_jobs=-1),
    X_tr_s, y_tr, cv=5, method="predict_proba"
)[:, 1]

# Target meta su train (il primario ha ragione?)
pred_train_dir = (p_primary_train_oof >= 0.5).astype(int)
meta_y_tr  = (pred_train_dir == y_tr).astype(int)

# Feature augmentate per meta-labeler
def augment_for_meta(X_s, p_primary):
    conf = np.abs(p_primary - 0.5)
    return np.column_stack([X_s, p_primary, conf])

X_tr_meta  = augment_for_meta(X_tr_s,  p_primary_train_oof)
X_val_meta = augment_for_meta(X_val_s, p_primary_val)
X_te_meta  = augment_for_meta(X_te_s,  p_primary_te)

lr_meta = LogisticRegression(C=0.1, class_weight="balanced", max_iter=500, random_state=42)
lr_meta.fit(X_tr_meta, meta_y_tr)

p_meta_val = lr_meta.predict_proba(X_val_meta)[:, 1]
p_meta_te  = lr_meta.predict_proba(X_te_meta)[:, 1]

# Score finale = p_primary × p_meta  (trade solo se ENTRAMBI concordano)
score_val = p_primary_val * p_meta_val
score_te  = p_primary_te  * p_meta_te

print(f"   Meta-Label LR — AUC val (meta): {roc_auc_score(y_val, p_meta_val):.4f}")
print(f"   Score composito (p_prim × p_meta) — range val: [{score_val.min():.4f}, {score_val.max():.4f}]")

# ─────────────────────────────────────────────────────────────────────────────
# 6. BACKTESTING OTTIMIZZATO (con regime filter)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[5/6] Calibrazione soglia + backtest ...")

def backtest_strategy(df_slice, score_long, score_short, thr_long, thr_short, fee_rt, vol_tgt_ann):
    """Backtest vettorizzato con vol-targeting e regime filter."""
    n = len(df_slice)
    close = df_slice["Close"].values.astype(float)
    low_vol = df_slice["low_vol_regime"].values
    vol40 = df_slice["vol_regime"].values

    # Vol-scaling: target annuo / (sigma_attuale * sqrt(bars_per_year))
    sigma = vol40 + 1e-12
    vol_scalar = (vol_tgt_ann / np.sqrt(BARS_PER_YEAR)) / sigma
    vol_scalar = np.clip(vol_scalar, 0.05, 3.0)

    # Segnale direzionale
    raw_pos = np.zeros(n)
    raw_pos[score_long  >= thr_long]  =  1.0  # LONG
    raw_pos[score_short >= thr_short] = -1.0  # SHORT (usa score_short = 1 - score_long)

    # Regime filter: azzera se high-vol
    raw_pos = raw_pos * low_vol

    # Applica vol-scaling solo quando in posizione
    pos_scaled = np.where(raw_pos != 0, raw_pos * vol_scalar, 0.0)
    pos_scaled = np.clip(pos_scaled, -3.0, 3.0)

    # Returns
    fwd_r = np.zeros(n)
    fwd_r[:-1] = (close[1:] / close[:-1]) - 1.0
    gross = pos_scaled * fwd_r

    # Fee: solo su cambio segnale discreto (non su vol-scaling continuo)
    signal_change = np.abs(np.diff(raw_pos, prepend=0.0))
    fee = signal_change * fee_rt

    net = gross - fee
    equity = (1 + net).cumprod()

    total_ret = equity[-1] - 1.0
    n_trades = int((signal_change > 0).sum())
    pct_active = float((raw_pos != 0).mean())
    running_max = np.maximum.accumulate(equity)
    drawdown = equity / running_max - 1.0
    max_dd = float(drawdown.min())

    # Sharpe annualizzato
    if net.std() > 1e-10:
        sharpe = float((net.mean() / net.std()) * np.sqrt(BARS_PER_YEAR))
    else:
        sharpe = -99.0

    return {
        "total_ret": total_ret,
        "sharpe": sharpe,
        "max_dd": max_dd,
        "n_trades": n_trades,
        "pct_active": pct_active,
        "equity": equity,
        "net_ret": net,
    }

# Calibrazione soglia su validation
df_val_reg = df_val.copy().reset_index(drop=True)
df_val_reg["low_vol_regime"] = df_val_reg["low_vol_regime"].values
df_val_reg["vol_regime"] = df_val_reg["vol_regime"].values

best_sharpe = -99.0
best_params = None

thr_grid = np.arange(0.25, 0.70, 0.05)
vt_grid  = [0.10, 0.15, 0.20]

for thr_l in thr_grid:
    thr_s = 1 - thr_l   # simmetrica: score basso → SHORT
    for vt in vt_grid:
        r = backtest_strategy(
            df_val_reg,
            score_val, 1 - score_val,  # per short: score basso = alta prob SHORT
            thr_l, thr_l,
            FEE_RT, vt
        )
        if r["n_trades"] >= 10 and r["max_dd"] > -0.40:
            if r["sharpe"] > best_sharpe:
                best_sharpe = r["sharpe"]
                best_params = {"thr_long": thr_l, "thr_short": thr_l, "vol_target": vt}

if best_params is None:
    best_params = {"thr_long": 0.45, "thr_short": 0.45, "vol_target": 0.15}
    print("   ⚠️  Nessun parametro soddisfa i vincoli su val → uso default")
else:
    print(f"   ✅ Parametri ottimali (val):  thr={best_params['thr_long']:.2f}  vol_tgt={best_params['vol_target']:.2f}  IS_Sharpe={best_sharpe:.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. TEST FINALE (parametri congelati)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n[6/6] Test finale (parametri congelati da validation) ...")

df_te_reg = df_te.copy().reset_index(drop=True)
res_test = backtest_strategy(
    df_te_reg,
    score_te, 1 - score_te,
    best_params["thr_long"], best_params["thr_short"],
    FEE_RT, best_params["vol_target"]
)

# ─────────────────────────────────────────────────────────────────────────────
# REPORT FINALE
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"  📊  RISULTATI FINALI — Stage 4 (5-min + MetaLabel + RegimeFilter)")
print(f"{'='*70}")
print(f"  Orizzonte previsione:  {H_BARS} barre × {RESAMPLE_MIN}min = {H_BARS*RESAMPLE_MIN}min")
print(f"  Fee round-trip:        {FEE_RT*100:.2f}%")
print(f"  Vol target annuo:      {best_params['vol_target']*100:.0f}%")
print(f"  Soglia score:          {best_params['thr_long']:.2f}")
print(f"  Regime filter:         vol < {VOL_REGIME_MULT}× mediana")
print(f"{'─'*70}")
print(f"  AUC modello primario:  {auc_primary_te:.4f}")
print(f"{'─'*70}")
print(f"  Total Return (OOS):    {res_test['total_ret']:+.2%}")
print(f"  Sharpe annualizzato:   {res_test['sharpe']:+.3f}")
print(f"  Max Drawdown:          {res_test['max_dd']:.2%}")
print(f"  N° trade:              {res_test['n_trades']:,}")
print(f"  % barre attive:        {res_test['pct_active']:.1%}")
print(f"{'='*70}")

# ── Grafici ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle(
    f"Stage 4 — 5min | Meta-Label | Regime Filter\n"
    f"Sharpe={res_test['sharpe']:+.2f} | Return={res_test['total_ret']:+.2%} | MaxDD={res_test['max_dd']:.2%}",
    fontsize=13, fontweight="bold"
)

dates_te = pd.to_datetime(df_te_reg["Date"].values)

# Panel 1: Equity curve
axes[0].plot(dates_te, res_test["equity"], color="steelblue", lw=1.4, label="Strategia")
axes[0].axhline(1.0, color="gray", lw=0.8, ls="--")
axes[0].set_ylabel("Equity (1=start)")
axes[0].legend(loc="lower left")
axes[0].set_title("Equity Curve (OOS Test)")

# Panel 2: Drawdown
dd = res_test["equity"] / np.maximum.accumulate(res_test["equity"]) - 1
axes[1].fill_between(dates_te, dd, 0, color="salmon", alpha=0.7)
axes[1].axhline(-0.20, color="orange", ls="--", lw=1.0, label="−20%")
axes[1].set_ylabel("Drawdown")
axes[1].legend(loc="lower left")
axes[1].set_title("Drawdown")

# Panel 3: Distribuzione score composito
axes[2].hist(score_te, bins=60, color="teal", alpha=0.7, edgecolor="white", lw=0.3)
axes[2].axvline(best_params["thr_long"],  color="green",  ls="--", lw=1.5, label=f"Soglia LONG  {best_params['thr_long']:.2f}")
axes[2].axvline(1-best_params["thr_short"], color="red", ls="--", lw=1.5, label=f"Soglia SHORT {1-best_params['thr_short']:.2f}")
axes[2].set_xlabel("Score composito (p_primary × p_meta)")
axes[2].set_ylabel("Count")
axes[2].legend()
axes[2].set_title("Distribuzione Score (Test)")

plt.tight_layout()
out_img = Path.home() / "Desktop" / "bitcoin" / "stage4_results.png"
plt.savefig(out_img, dpi=120, bbox_inches="tight")
plt.show()
print(f"\n  💾 Grafico salvato: {out_img.name}")


  STAGE 4 — Soluzione: Resample 5m + MetaLabel + Regime Filter

[1/6] Caricamento dati da: BTCUSD_1m_Binance.xlsx
   1-min bars: 999,999  →  5-min bars: 200,008

[2/6] Feature engineering su barre 5-min ...
   Barre totali: 148,513
   Barre etichettate: 148,513 (100.0%)
   Barre low-vol regime: 116,544 (78.5%)

[3/6] Split temporale (70/15/15) ...
   1-min bars: 999,999  →  5-min bars: 200,008

[2/6] Feature engineering su barre 5-min ...
   Barre totali: 148,513
   Barre etichettate: 148,513 (100.0%)
   Barre low-vol regime: 116,544 (78.5%)

[3/6] Split temporale (70/15/15) ...
   Train:  103,959 | Val: 22,277 | Test: 22,277
   Feature: 30

[4/6] Training modelli (RF primario + Meta-label LR) ...
   Train:  103,959 | Val: 22,277 | Test: 22,277
   Feature: 30

[4/6] Training modelli (RF primario + Meta-label LR) ...
   RF Primario — AUC val: 0.6095  |  AUC test: 0.5790
   RF Primario — AUC val: 0.6095  |  AUC test: 0.5790
   Meta-Label LR — AUC val (meta): 0.4943
   Score composito (p_

In [14]:
#funzionanteeeeeeeeeeeeeeeee
# ==============================================================================
# STAGE 5 — DIAGNOSI + TRIPLE BARRIER + HOLD-TO-MATURITY BACKTEST
# ==============================================================================
# Approccio radicalmente diverso basato su Lopez de Prado (AFML 2018)
# e risultati pratici dai paper più citati su crypto ML trading.
#
# DIAGNOSI PRIMA: verifichiamo l'edge REALE della previsione PRIMA del backtest.
# ==============================================================================

import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

print("=" * 70)
print("  STAGE 5 — Diagnosi Edge + Triple Barrier + Hold-to-Maturity")
print("=" * 70)

# Riutilizza il df5 già costruito nello Stage 4 (barre 5-min)
# Ci serve df, df_tr, df_val, df_te, X_tr_s, X_val_s, X_te_s, feat_cols
# già definiti sopra (kernel memory)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 0: DIAGNOSI — Edge reale del modello
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"  STEP 0: DIAGNOSI EDGE REALE")
print(f"{'─'*70}")

# La previsione p_primary_te è la prob LONG per ogni barra 5-min del test
# fwd_ret_h è il return a 30 min (H_BARS=6 × 5min)
# Dobbiamo verificare: le barre con p_primary_te alto hanno davvero return positivo?

# Decili di p_primary_te vs return medio a 30min
fwd_30min = df_te["fwd_ret_h"].values  # return effettivo a 30min
p = p_primary_te  # già in memoria

decile_edges = np.percentile(p, np.linspace(0, 100, 11))
print(f"\n  Decile analisi: p_model (test) vs return a 30min")
print(f"  {'Decile':>7} {'p_range':>15} {'N':>6} {'Avg_ret_30m':>13} {'Pos%':>7}")
print(f"  {'─'*60}")

decile_results = []
for i in range(10):
    lo, hi = decile_edges[i], decile_edges[i+1]
    mask = (p >= lo) & (p <= hi + 1e-9)
    n_in = mask.sum()
    if n_in > 0:
        avg_r = fwd_30min[mask].mean()
        pos_pct = (fwd_30min[mask] > 0).mean()
        label = f"D{i+1:02d}"
        flag = "✅" if avg_r > 0 and i >= 7 else ("🔻" if avg_r < 0 and i <= 2 else "  ")
        print(f"  {label:>7} [{lo:.3f},{hi:.3f}] {n_in:>6,}  {avg_r*100:>+10.4f}%  {pos_pct:>6.1%} {flag}")
        decile_results.append({"decile": i+1, "lo": lo, "hi": hi, "n": n_in, "avg_ret": avg_r, "pos_pct": pos_pct})

print(f"\n  💡 Se D10 >> D1 il modello ha edge direzionale reale.")
df_diag = pd.DataFrame(decile_results)
edge_spread = df_diag.iloc[-1]["avg_ret"] - df_diag.iloc[0]["avg_ret"]
print(f"  Edge spread (D10 - D1): {edge_spread*100:+.4f}% per barra 5-min")
print(f"  Edge annualizzato (×{BARS_PER_YEAR:.0f} barre/anno): {edge_spread*BARS_PER_YEAR*100:+.1f}% (teorico, senza costi)")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: TRIPLE BARRIER (Lopez de Prado, AFML cap.3)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"  STEP 1: TRIPLE BARRIER LABELING")
print(f"{'─'*70}")
# Le 3 barriere:
#   - UP:   price + PT × atr  → take profit (label +1)
#   - DOWN: price - SL × atr  → stop loss   (label -1 → y=0)
#   - TIME: max_hold barre     → scadenza    (label basato su direction)
#
# Usiamo l'ATR normalizzato × close come distanza barriera.

def triple_barrier_labels(close_arr, atr_arr, pt=1.5, sl=1.0, max_hold=12):
    """
    close_arr: array prezzi close
    atr_arr: array ATR assoluto (non norm)
    pt: take-profit multiplo (× ATR)
    sl: stop-loss multiplo (× ATR)
    max_hold: massimo barre da tenere
    Returns: labels array (1=long win, 0=short win/stop, NaN=ambiguous)
    """
    n = len(close_arr)
    labels = np.full(n, np.nan)

    for i in range(n - max_hold):
        c0 = close_arr[i]
        atr = atr_arr[i]
        if atr <= 0 or np.isnan(atr):
            continue

        tp_up   = c0 + pt * atr
        tp_down = c0 - sl * atr

        hit_up   = np.inf
        hit_down = np.inf

        for j in range(i+1, min(i+max_hold+1, n)):
            c_j = close_arr[j]
            if c_j >= tp_up:
                hit_up = j - i
                break
            if c_j <= tp_down:
                hit_down = j - i
                break

        if hit_up < hit_down:
            labels[i] = 1.0  # Take Profit LONG colpito prima
        elif hit_down < hit_up:
            labels[i] = 0.0  # Stop Loss colpito (o TP SHORT)
        else:
            # Nessuna barriera colpita → label da segno del return a scadenza
            ret_h = close_arr[min(i+max_hold, n-1)] / c0 - 1.0
            if ret_h > 0.0005:
                labels[i] = 1.0
            elif ret_h < -0.0005:
                labels[i] = 0.0
            # else: NaN = no trade

    return labels

# Calcola ATR assoluto su barre 5-min del dataset completo
atr_abs = df["atr_14_norm"].values * df["Close"].values  # de-normalizzato

# Applica Triple Barrier su TUTTO il dataset (poi splitto)
print("  Calcolo Triple Barrier labels (PT=1.5×ATR, SL=1.0×ATR, max_hold=12 barre=1h)...")
close_all = df["Close"].values.astype(float)
tb_labels = triple_barrier_labels(close_all, atr_abs, pt=1.5, sl=1.0, max_hold=12)

df["tb_label"] = tb_labels
n_tb = (~np.isnan(tb_labels)).sum()
n_tb_long  = (tb_labels == 1).sum()
n_tb_short = (tb_labels == 0).sum()
print(f"  TB labels: {n_tb:,} | Long(TP hit): {n_tb_long:,} ({n_tb_long/n_tb*100:.1f}%) | Short(SL hit): {n_tb_short:,} ({n_tb_short/n_tb*100:.1f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: TRAIN RF SU TRIPLE BARRIER LABELS
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"  STEP 2: RF su Triple Barrier Labels")
print(f"{'─'*70}")

# Split temporale sulle righe con TB label non-NaN
df_tb = df.dropna(subset=["tb_label"]).reset_index(drop=True)
df_tb["tb_label"] = df_tb["tb_label"].astype(int)

drop_cols_tb = ["Date", "target", "fwd_ret_h", "low_vol_regime", "vol_regime",
                "tb_label", "Open", "High", "Low", "Close", "Volume", "logret"]
feat_cols_tb = [c for c in df_tb.columns if c not in drop_cols_tb]

n_tb_total = len(df_tb)
i_tr_tb  = int(n_tb_total * 0.70)
i_val_tb = int(n_tb_total * 0.85)

df_tb_tr  = df_tb.iloc[:i_tr_tb].copy()
df_tb_val = df_tb.iloc[i_tr_tb:i_val_tb].copy()
df_tb_te  = df_tb.iloc[i_val_tb:].copy()

for _d in [df_tb_tr, df_tb_val, df_tb_te]:
    _d[feat_cols_tb] = _d[feat_cols_tb].replace([np.inf, -np.inf], np.nan).fillna(0.0)

X_tb_tr  = df_tb_tr[feat_cols_tb].values
X_tb_val = df_tb_val[feat_cols_tb].values
X_tb_te  = df_tb_te[feat_cols_tb].values
y_tb_tr  = df_tb_tr["tb_label"].values
y_tb_val = df_tb_val["tb_label"].values
y_tb_te  = df_tb_te["tb_label"].values

scaler_tb = StandardScaler()
X_tb_tr_s  = scaler_tb.fit_transform(X_tb_tr)
X_tb_val_s = scaler_tb.transform(X_tb_val)
X_tb_te_s  = scaler_tb.transform(X_tb_te)

rf_tb = RandomForestClassifier(
    n_estimators=500, max_depth=10, min_samples_leaf=80,
    max_features="sqrt", class_weight="balanced",
    random_state=42, n_jobs=-1
)
print("  Training RF su TB labels...")
rf_tb.fit(X_tb_tr_s, y_tb_tr)

p_tb_val = rf_tb.predict_proba(X_tb_val_s)[:, 1]
p_tb_te  = rf_tb.predict_proba(X_tb_te_s)[:, 1]

auc_tb_val = roc_auc_score(y_tb_val, p_tb_val)
auc_tb_te  = roc_auc_score(y_tb_te,  p_tb_te)
print(f"  RF Triple Barrier — AUC val: {auc_tb_val:.4f}  |  AUC test: {auc_tb_te:.4f}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: HOLD-TO-MATURITY BACKTEST (entra, tieni 12 barre = 1h, esci)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"  STEP 3: Hold-To-Maturity Backtest (soglia alta, 0 overlap)")
print(f"{'─'*70}")
# Logica: entra SOLO se p_tb >= THR_ENTRY
# Tieni per esattamente max_hold barre (o fino a TP/SL)
# NO sovrapposizione: dopo l'uscita si aspetta il prossimo segnale
# COSTO: pagato UNA VOLTA all'entrata + UNA VOLTA all'uscita = fee RT

HOLD_BARS = 12       # 12 × 5min = 1h
# --- Binance SPOT con sconto BNB: 0.075% entry + 0.075% exit = 0.15% round-trip
FEE_ENTRY = 0.00075
FEE_EXIT  = 0.00075
FEE_RT_TB = FEE_ENTRY + FEE_EXIT   # = 0.0015

def htm_backtest(df_slice, p_signal, thr_entry, hold_bars, fee_rt, atr_col="atr_14_norm", pt_mult=1.5, sl_mult=1.0):
    """Hold-to-maturity: entra quando segnale, tieni H barre o hit TP/SL."""
    close = df_slice["Close"].values.astype(float)
    atr_norm = df_slice[atr_col].values.astype(float)
    n = len(close)

    pnl_list = []
    trade_dates = []
    i = 0
    n_tp = n_sl = n_time = 0

    while i < n - hold_bars:
        if p_signal[i] >= thr_entry:
            # ENTRA LONG
            entry_price = close[i]
            atr_val = atr_norm[i] * entry_price
            tp_price = entry_price * (1 + pt_mult * atr_norm[i])
            sl_price = entry_price * (1 - sl_mult * atr_norm[i])

            exit_price = close[min(i + hold_bars, n-1)]
            exit_reason = "TIME"

            for j in range(i+1, min(i+hold_bars+1, n)):
                if close[j] >= tp_price:
                    exit_price = tp_price
                    exit_reason = "TP"
                    break
                if close[j] <= sl_price:
                    exit_price = sl_price
                    exit_reason = "SL"
                    break

            # P&L del trade
            gross_ret = (exit_price / entry_price) - 1.0
            net_ret = gross_ret - fee_rt  # fee pagata UNA VOLTA (round-trip)
            pnl_list.append(net_ret)
            trade_dates.append(df_slice["Date"].iloc[i])

            if exit_reason == "TP": n_tp += 1
            elif exit_reason == "SL": n_sl += 1
            else: n_time += 1

            # Salta avanti di hold_bars (no overlap)
            i += hold_bars + 1
        else:
            i += 1

    if len(pnl_list) == 0:
        return {"n_trades": 0, "total_ret": 0.0, "sharpe": -99.0, "max_dd": 0.0,
                "win_rate": 0.0, "avg_pnl": 0.0, "pnl_list": [], "trade_dates": []}

    pnl_arr = np.array(pnl_list)
    equity = (1 + pnl_arr).cumprod()
    total_ret = equity[-1] - 1.0
    win_rate = (pnl_arr > 0).mean()
    avg_pnl = pnl_arr.mean()

    # Sharpe annualizzato (per trade): trades_per_year = BARS_PER_YEAR / hold_bars
    # anni effettivi nel sample (barre 5-min)
    years_in_sample = (len(df_slice) / BARS_PER_YEAR)
    trades_per_year = len(pnl_arr) / max(years_in_sample, 1e-9)
    sharpe = (avg_pnl / pnl_arr.std()) * np.sqrt(trades_per_year)
    if pnl_arr.std() > 1e-10:
        sharpe = (avg_pnl / pnl_arr.std()) * np.sqrt(trades_per_year)
    else:
        sharpe = -99.0

    running_max = np.maximum.accumulate(equity)
    max_dd = float((equity / running_max - 1).min())

    return {
        "n_trades": len(pnl_list),
        "n_tp": n_tp, "n_sl": n_sl, "n_time": n_time,
        "total_ret": total_ret,
        "sharpe": sharpe,
        "max_dd": max_dd,
        "win_rate": win_rate,
        "avg_pnl": avg_pnl,
        "pnl_list": pnl_list,
        "trade_dates": trade_dates,
        "equity": equity,
    }

# Sweep soglia su validation
print(f"\n  Calibrazione soglia su validation...")
print(f"  {'THR':>6} {'N':>6} {'WinRate':>8} {'AvgPnL%':>9} {'Sharpe':>8} {'TotalRet%':>11}")
print(f"  {'─'*60}")

best_htms = -99.0
best_thr_htm = 0.65

for thr in np.arange(0.50, 0.90, 0.05):
    r = htm_backtest(df_tb_val.reset_index(drop=True), p_tb_val, thr, HOLD_BARS, FEE_RT_TB)
    if r["n_trades"] >= 5:
        flag = "✅" if r["sharpe"] > 0 else ""
        print(f"  {thr:.2f}   {r['n_trades']:>6}  {r['win_rate']:>7.1%}  {r['avg_pnl']*100:>+8.3f}%  {r['sharpe']:>7.3f}  {r['total_ret']*100:>+10.2f}% {flag}")
        if r["sharpe"] > best_htms:
            best_htms = r["sharpe"]
            best_thr_htm = thr

print(f"\n  ✅ Soglia ottimale (val): {best_thr_htm:.2f}  IS_Sharpe: {best_htms:.3f}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: RISULTATO FINALE OOS
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"  STEP 4: RISULTATO FINALE OOS (soglia congelata {best_thr_htm:.2f})")
print(f"{'─'*70}")

res_htm = htm_backtest(df_tb_te.reset_index(drop=True), p_tb_te, best_thr_htm, HOLD_BARS, FEE_RT_TB)

print(f"\n{'='*70}")
print(f"  📊  STAGE 5 — Hold-To-Maturity Backtest — RISULTATI OOS")
print(f"{'='*70}")
print(f"  Modello:          RF su Triple Barrier (PT=1.5×ATR, SL=1.0×ATR, T=12bar)")
print(f"  Frequenza barre:  5-min  |  Hold: {HOLD_BARS} barre = {HOLD_BARS*5}min")
print(f"  Fee round-trip:   {FEE_RT_TB*100:.2f}%  |  Soglia entry: {best_thr_htm:.2f}")
print(f"{'─'*70}")
print(f"  AUC RF (TB):      {auc_tb_te:.4f}")
print(f"{'─'*70}")

if res_htm["n_trades"] > 0:
    print(f"  N° trade:         {res_htm['n_trades']}")
    print(f"    ├─ Take Profit: {res_htm['n_tp']}  ({res_htm['n_tp']/res_htm['n_trades']*100:.1f}%)")
    print(f"    ├─ Stop Loss:   {res_htm['n_sl']}  ({res_htm['n_sl']/res_htm['n_trades']*100:.1f}%)")
    print(f"    └─ Time Exit:   {res_htm['n_time']}  ({res_htm['n_time']/res_htm['n_trades']*100:.1f}%)")
    print(f"  Win Rate:         {res_htm['win_rate']:.2%}")
    print(f"  Avg P&L/trade:    {res_htm['avg_pnl']*100:+.4f}%")
    print(f"  Total Return:     {res_htm['total_ret']*100:+.2f}%")
    print(f"  Sharpe (ann.):    {res_htm['sharpe']:+.3f}")
    print(f"  Max Drawdown:     {res_htm['max_dd']:.2%}")
else:
    print(f"  ⚠️  Nessun trade generato con soglia {best_thr_htm:.2f} nel test set.")

print(f"{'='*70}")

# ─────────────────────────────────────────────────────────────────────────────
# GRAFICO FINALE
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    f"Stage 5 — Triple Barrier + HTM Backtest\n"
    f"AUC={auc_tb_te:.3f} | N_trade={res_htm['n_trades']} | Sharpe={res_htm['sharpe']:+.2f} | Return={res_htm['total_ret']*100:+.1f}%",
    fontsize=12, fontweight="bold"
)

# Panel 1: Equity Curve (per trade)
ax = axes[0, 0]
if res_htm["n_trades"] > 0:
    ax.plot(res_htm["equity"], color="steelblue", lw=1.5, marker="o", markersize=3)
    ax.axhline(1.0, color="gray", lw=0.8, ls="--")
ax.set_title("Equity Curve (per trade)")
ax.set_xlabel("Trade #")
ax.set_ylabel("Equity")

# Panel 2: Distribuzione P&L per trade
ax = axes[0, 1]
if res_htm["n_trades"] > 0:
    pnl_arr = np.array(res_htm["pnl_list"])
    ax.hist(pnl_arr * 100, bins=30, color="teal", alpha=0.75, edgecolor="white")
    ax.axvline(0, color="red", lw=1.0, ls="--")
    ax.axvline(pnl_arr.mean() * 100, color="green", lw=1.5, ls="-", label=f"Mean={pnl_arr.mean()*100:+.3f}%")
    ax.legend()
ax.set_title("Distribuzione P&L per trade (%)")
ax.set_xlabel("Return netto (%)")
ax.set_ylabel("Count")

# Panel 3: Decile analisi (edge diagnostics)
ax = axes[1, 0]
if len(df_diag) > 0:
    colors = ["#d62728" if r < 0 else "#2ca02c" for r in df_diag["avg_ret"]]
    bars = ax.bar(df_diag["decile"], df_diag["avg_ret"] * 100, color=colors, alpha=0.8, edgecolor="white")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(df_diag["decile"])
    ax.set_xticklabels([f"D{d}" for d in df_diag["decile"]], fontsize=9)
ax.set_title("Decile p_model vs Avg Return 30min (diagnostica edge)")
ax.set_xlabel("Decile score (D1=basso, D10=alto)")
ax.set_ylabel("Avg Return 30min (%)")

# Panel 4: Distribuzione score test
ax = axes[1, 1]
ax.hist(p_tb_te, bins=50, color="purple", alpha=0.7, edgecolor="white")
ax.axvline(best_thr_htm, color="green", lw=2, ls="--", label=f"Soglia LONG {best_thr_htm:.2f}")
ax.axvline(1-best_thr_htm, color="red", lw=2, ls="--", label=f"Soglia SHORT {1-best_thr_htm:.2f}")
ax.legend()
ax.set_title("Distribuzione Score RF (Triple Barrier)")
ax.set_xlabel("p(long)")
ax.set_ylabel("Count")

plt.tight_layout()
out_img5 = Path.home() / "Desktop" / "bitcoin" / "stage5_results.png"
plt.savefig(out_img5, dpi=120, bbox_inches="tight")
plt.show()
print(f"\n  💾 Grafico salvato: {out_img5.name}")


  STAGE 5 — Diagnosi Edge + Triple Barrier + Hold-to-Maturity

──────────────────────────────────────────────────────────────────────
  STEP 0: DIAGNOSI EDGE REALE
──────────────────────────────────────────────────────────────────────

  Decile analisi: p_model (test) vs return a 30min
   Decile         p_range      N   Avg_ret_30m    Pos%
  ────────────────────────────────────────────────────────────
      D01 [0.272,0.396]  2,228     -0.0399%   40.6% 🔻
      D02 [0.396,0.430]  2,228     -0.0127%   42.7% 🔻
      D03 [0.430,0.457]  2,227     +0.0226%   47.9%   
      D04 [0.457,0.481]  2,228     +0.0395%   52.6%   
      D05 [0.481,0.501]  2,228     +0.0488%   53.9%   
      D06 [0.501,0.521]  2,228     +0.0020%   51.6%   
      D07 [0.521,0.541]  2,228     +0.0313%   55.7%   
      D08 [0.541,0.563]  2,227     +0.0181%   56.6% ✅
      D09 [0.563,0.590]  2,228     +0.0531%   60.0% ✅
      D10 [0.590,0.707]  2,228     +0.0626%   64.6% ✅

  💡 Se D10 >> D1 il modello ha edge direzionale r

## 🎯 Ottimizzazione per Utility Economica + Meta-Labeling

Invece di ottimizzare per **accuracy/AUC**, qui ottimizziamo direttamente per **Sharpe ratio** o **P&L netto** (al netto delle fee).

### Approccio
1. **Economic Scorer**: funzione obiettivo custom che calcola Sharpe/P&L netto simulando la strategia su ogni configurazione di iperparametri.
2. **Meta-Labeling**: un modello secondario (filtro) che impara **quando** il modello primario ha davvero edge — riducendo i false positive costosi.
   - Modello primario → genera segnale (LONG/SHORT/FLAT)
   - Modello secondario (meta-label) → decide se **eseguire** il segnale o ignorarlo (binary: 1=entra, 0=skip)


In [ ]:
# ==============================================================================
# STAGE 6 — OTTIMIZZAZIONE PER UTILITY ECONOMICA + META-LABELING
# ==============================================================================
# Obiettivo: ottimizzare Sharpe / P&L netto invece di accuracy/AUC.
# Meta-labeling: modello secondario che decide SE entrare nel trade.
# ==============================================================================

import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from itertools import product

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)

# ──────────────────────────────────────────────────────────────────────────────
# PARAMETRI GLOBALI
# ──────────────────────────────────────────────────────────────────────────────
RESAMPLE_MIN  = 5
H_BARS        = 6          # orizzonte: 6 × 5min = 30min
FEE_RT        = 0.0010     # round-trip 0.10%
BARS_PER_YEAR = 252 * 24 * 60 // RESAMPLE_MIN

print("=" * 70)
print("  STAGE 6 — Economic Utility Optimization + Meta-Labeling")
print("=" * 70)

# ──────────────────────────────────────────────────────────────────────────────
# 1. CARICA E RICAMPIONA
# ──────────────────────────────────────────────────────────────────────────────
raw_path = Path.home() / "Desktop" / "bitcoin" / "BTCUSD_1m_Binance.xlsx"
raw = pd.read_excel(raw_path)
raw.columns = [str(c).strip() for c in raw.columns]
col_map = {c.lower(): c for c in raw.columns}
raw = raw.rename(columns={
    col_map["open time"]: "Date", col_map["open"]:  "Open",
    col_map["high"]:  "High",    col_map["low"]:   "Low",
    col_map["close"]: "Close",   col_map["volume"]: "Volume",
})
raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
raw = raw.dropna(subset=["Date"]).sort_values("Date").drop_duplicates("Date", keep="last").set_index("Date")
for c in ["Open","High","Low","Close","Volume"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")
raw = raw.dropna()

df = raw.resample(f"{RESAMPLE_MIN}min").agg(
    Open=("Open","first"), High=("High","max"),
    Low=("Low","min"),     Close=("Close","last"),
    Volume=("Volume","sum")
).dropna().reset_index()
print(f"   5-min bars: {len(df):,}")

# ──────────────────────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ──────────────────────────────────────────────────────────────────────────────
eps = 1e-12
df["logret"] = np.log(df["Close"] + eps).diff()
for n in [2, 3, 5, 10, 20, 40]:
    df[f"mom_{n}"] = df["Close"].pct_change(n)
for n in [10, 20, 40]:
    df[f"vol_{n}"] = df["logret"].rolling(n, min_periods=n).std()
delta = df["Close"].diff()
ag = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
al = (-delta).clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
df["rsi"] = 100 - 100 / (1 + ag / (al + eps))
df["macd_hist"] = (df["Close"].ewm(span=12,adjust=False).mean()
                  - df["Close"].ewm(span=26,adjust=False).mean()
                  - df["Close"].ewm(span=9,adjust=False).mean())
for n in [20, 40]:
    mid = df["Close"].rolling(n).mean()
    std = df["Close"].rolling(n).std()
    df[f"bb_z_{n}"] = (df["Close"] - mid) / (std + eps)
df["vol_ratio"] = df["vol_10"] / (df["vol_40"] + eps)

# TARGET: rendimento netto a H_BARS barre
df["fwd_ret"] = df["Close"].shift(-H_BARS) / df["Close"] - 1.0
df["label"]   = (df["fwd_ret"] > FEE_RT / 2).astype(int)   # 1=LONG, 0=SHORT/FLAT

FEATURES = [c for c in df.columns if c.startswith(("mom_","vol_","rsi","macd","bb_z","logret","vol_ratio"))]
df = df.dropna(subset=FEATURES + ["fwd_ret", "label"]).reset_index(drop=True)

# ──────────────────────────────────────────────────────────────────────────────
# 3. SPLIT TRAIN / VAL / TEST  (60 / 20 / 20 — walk-forward)
# ──────────────────────────────────────────────────────────────────────────────
n = len(df)
i_tr  = int(n * 0.60)
i_val = int(n * 0.80)

df_tr  = df.iloc[:i_tr].copy()
df_val = df.iloc[i_tr:i_val].copy()
df_te  = df.iloc[i_val:].copy()

X_tr,  y_tr  = df_tr[FEATURES].values,  df_tr["label"].values
X_val, y_val = df_val[FEATURES].values, df_val["label"].values
X_te,  y_te  = df_te[FEATURES].values,  df_te["label"].values

print(f"   Train: {len(df_tr):,} | Val: {len(df_val):,} | Test: {len(df_te):,}")


# ──────────────────────────────────────────────────────────────────────────────
# 4. FUNZIONE DI UTILITÀ ECONOMICA
#    Simula la strategia e restituisce Sharpe netto annualizzato.
#    Parametri ottimizzabili: threshold (quanto deve essere sicuro il modello)
#    e size (frazione del capitale per trade).
# ──────────────────────────────────────────────────────────────────────────────
def economic_utility(proba: np.ndarray, fwd_ret: np.ndarray,
                     threshold: float = 0.55, size: float = 1.0,
                     fee_rt: float = FEE_RT) -> dict:
    """
    Calcola Sharpe netto e P&L netto dati:
    - proba    : probabilità p(LONG) dal modello primario
    - fwd_ret  : rendimenti futuri realizzati
    - threshold: soglia minima per entrare LONG (1-thr per SHORT)
    - size     : frazione capitale per trade (es. 0.5 = half kelly)
    """
    pos = np.zeros(len(proba))
    pos[proba >= threshold]       =  size   # LONG
    pos[proba <= (1 - threshold)] = -size   # SHORT

    gross = pos * fwd_ret
    # fee solo a cambio posizione
    fee = np.abs(np.diff(pos, prepend=0.0)) * fee_rt
    net = gross - fee

    n_trades = int((np.abs(np.diff(pos, prepend=0.0)) > 0).sum())
    total_ret = float((1 + net).prod() - 1)
    std = float(net.std(ddof=1))
    sharpe = float(net.mean() / std * np.sqrt(BARS_PER_YEAR)) if std > 0 else 0.0
    return {"sharpe": sharpe, "total_ret": total_ret, "n_trades": n_trades, "net_rets": net}


# ──────────────────────────────────────────────────────────────────────────────
# 5. OTTIMIZZAZIONE IPERPARAMETRI RF CON OBIETTIVO = SHARPE
#    Grid search su (n_estimators, max_depth, min_samples_leaf, threshold)
#    → selezione in base a Sharpe su validation set
# ──────────────────────────────────────────────────────────────────────────────
print("\n[3/5] Grid search economica su validation set ...")

param_grid = {
    "n_estimators":     [100, 300],
    "max_depth":        [4, 6, 8],
    "min_samples_leaf": [20, 50],
    "threshold":        [0.52, 0.55, 0.58, 0.60],
}

best_sharpe_val = -np.inf
best_cfg        = None
best_proba_val  = None
best_model      = None
results_grid    = []

combos = list(product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["min_samples_leaf"],
    param_grid["threshold"],
))
print(f"   Combinazioni da testare: {len(combos)}")

for n_est, max_d, min_leaf, thr in combos:
    rf = RandomForestClassifier(
        n_estimators=n_est, max_depth=max_d,
        min_samples_leaf=min_leaf, n_jobs=-1,
        class_weight="balanced", random_state=42
    )
    rf.fit(X_tr, y_tr)
    p_val = rf.predict_proba(X_val)[:, 1]
    util  = economic_utility(p_val, df_val["fwd_ret"].values, threshold=thr)
    results_grid.append({
        "n_estimators": n_est, "max_depth": max_d,
        "min_samples_leaf": min_leaf, "threshold": thr,
        **{k: util[k] for k in ("sharpe","total_ret","n_trades")}
    })
    if util["sharpe"] > best_sharpe_val:
        best_sharpe_val = util["sharpe"]
        best_cfg  = dict(n_estimators=n_est, max_depth=max_d,
                         min_samples_leaf=min_leaf, threshold=thr)
        best_proba_val = p_val
        best_model     = rf

results_df = pd.DataFrame(results_grid).sort_values("sharpe", ascending=False)
print(f"\n   ✅ Best config (val Sharpe={best_sharpe_val:.3f}):")
for k, v in best_cfg.items():
    print(f"      {k}: {v}")

print("\n   Top 5 configurazioni per Sharpe:")
print(results_df.head(5).to_string(index=False))


# ──────────────────────────────────────────────────────────────────────────────
# 6. META-LABELING
#    Il modello primario (best_model) genera segnali (LONG/SHORT).
#    Il modello secondario (meta-labeler) impara a prevedere se
#    quel segnale sarà profittevole → binary outcome: 1=entra, 0=skip.
#
#    Features del meta-labeler:
#    - p_primary        : confidence del modello primario
#    - features originali
#    - volatilità regionale (contesto di regime)
# ──────────────────────────────────────────────────────────────────────────────
print("\n[4/5] Meta-labeling: addestramento modello secondario ...")

# --- Costruisci meta-label su TRAINING set ---
p_tr_primary = best_model.predict_proba(X_tr)[:, 1]
thr_best = best_cfg["threshold"]

# Segnale del primario (solo su barre dove entra)
signal_tr = np.where(p_tr_primary >= thr_best, 1,
            np.where(p_tr_primary <= 1 - thr_best, -1, 0))

# Meta-label: 1 se il segnale è stato profittevole (netto fee), 0 altrimenti
gross_tr = signal_tr * df_tr["fwd_ret"].values
meta_label_tr = ((gross_tr - FEE_RT) > 0).astype(int)

# Usa solo le barre dove il primario ha emesso un segnale (non flat)
active_tr = signal_tr != 0
X_meta_tr   = np.column_stack([X_tr[active_tr], p_tr_primary[active_tr]])
y_meta_tr   = meta_label_tr[active_tr]

# --- Stesso processo su VAL per test del meta-labeler ---
p_val_primary = best_proba_val  # già calcolato
signal_val = np.where(p_val_primary >= thr_best, 1,
             np.where(p_val_primary <= 1 - thr_best, -1, 0))
gross_val  = signal_val * df_val["fwd_ret"].values
meta_label_val = ((gross_val - FEE_RT) > 0).astype(int)
active_val = signal_val != 0
X_meta_val = np.column_stack([X_val[active_val], p_val_primary[active_val]])
y_meta_val = meta_label_val[active_val]

print(f"   Segnali attivi train:  {active_tr.sum():,} / {len(X_tr):,}")
print(f"   Segnali attivi val:    {active_val.sum():,} / {len(X_val):,}")

# Addestra meta-labeler (Gradient Boosting per catturare non-linearità)
scaler_meta = StandardScaler()
X_meta_tr_s = scaler_meta.fit_transform(X_meta_tr)
X_meta_val_s = scaler_meta.transform(X_meta_val)

meta_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05,
    subsample=0.8, random_state=42
)
meta_model.fit(X_meta_tr_s, y_meta_tr)

if len(np.unique(y_meta_val)) > 1:
    auc_meta = roc_auc_score(y_meta_val, meta_model.predict_proba(X_meta_val_s)[:, 1])
    print(f"   Meta-labeler AUC (val): {auc_meta:.4f}")
else:
    print("   Meta-labeler: classe unica su val, AUC non calcolabile.")


# ──────────────────────────────────────────────────────────────────────────────
# 7. VALUTAZIONE FINALE SU TEST SET
#    Confronto tra:
#    (A) RF primario ottimizzato per accuracy (soglia = 0.5)
#    (B) RF primario ottimizzato per Sharpe   (soglia = best_cfg["threshold"])
#    (C) RF primario + Meta-labeler (entra solo se meta_proba >= 0.5)
# ──────────────────────────────────────────────────────────────────────────────
print("\n[5/5] Valutazione su test set ...")

p_te = best_model.predict_proba(X_te)[:, 1]

# (A) Primario con soglia 0.50
util_A = economic_utility(p_te, df_te["fwd_ret"].values, threshold=0.50)

# (B) Primario con soglia ottimizzata economicamente
util_B = economic_utility(p_te, df_te["fwd_ret"].values, threshold=thr_best)

# (C) Primario + Meta-labeler
signal_te  = np.where(p_te >= thr_best, 1,
             np.where(p_te <= 1 - thr_best, -1, 0))
active_te  = signal_te != 0
X_meta_te  = np.column_stack([X_te[active_te], p_te[active_te]])
X_meta_te_s = scaler_meta.transform(X_meta_te)
meta_proba_te = meta_model.predict_proba(X_meta_te_s)[:, 1]

# Applica filtro meta: entra solo se meta_proba >= 0.5
pos_C = np.zeros(len(p_te))
active_idx = np.where(active_te)[0]
for i, idx in enumerate(active_idx):
    if meta_proba_te[i] >= 0.50:
        pos_C[idx] = float(signal_te[idx])

# Calcola utility manuale per (C)
gross_C = pos_C * df_te["fwd_ret"].values
fee_C   = np.abs(np.diff(pos_C, prepend=0.0)) * FEE_RT
net_C   = gross_C - fee_C
n_tr_C  = int((np.abs(np.diff(pos_C, prepend=0.0)) > 0).sum())
std_C   = net_C.std(ddof=1)
sharpe_C = float(net_C.mean() / std_C * np.sqrt(BARS_PER_YEAR)) if std_C > 0 else 0.0
total_C  = float((1 + net_C).prod() - 1)

print(f"\n{'='*65}")
print(f"  RISULTATI FINALI SU TEST SET")
print(f"{'='*65}")
print(f"  {'Strategia':<42} {'Sharpe':>8}  {'Return':>8}  {'#Trades':>8}")
print(f"  {'─'*62}")
print(f"  (A) RF primario  [thr=0.50, baseline accuracy]  "
      f"{util_A['sharpe']:>+8.3f}  {util_A['total_ret']*100:>+7.2f}%  {util_A['n_trades']:>7,}")
print(f"  (B) RF ottimizzato Sharpe [thr={thr_best:.2f}]           "
      f"{util_B['sharpe']:>+8.3f}  {util_B['total_ret']*100:>+7.2f}%  {util_B['n_trades']:>7,}")
print(f"  (C) RF + Meta-labeler    [thr={thr_best:.2f} + filtro]   "
      f"{sharpe_C:>+8.3f}  {total_C*100:>+7.2f}%  {n_tr_C:>7,}")
print(f"{'='*65}")

# ── Plot equity curves ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Confronto Strategie — Economic Utility Optimization + Meta-Labeling",
             fontsize=13, fontweight="bold")

dates_te = pd.to_datetime(df_te["Date"].values)
for ax, net_rets, label, color in [
    (axes[0], util_A["net_rets"], f"(A) Baseline thr=0.50\nSharpe={util_A['sharpe']:+.3f}", "steelblue"),
    (axes[1], util_B["net_rets"], f"(B) Ottimiz. Sharpe thr={thr_best:.2f}\nSharpe={util_B['sharpe']:+.3f}", "darkorange"),
    (axes[2], net_C,              f"(C) RF + Meta-label\nSharpe={sharpe_C:+.3f}", "green"),
]:
    eq = (1 + net_rets).cumprod()
    ax.plot(dates_te[:len(eq)], eq, color=color, lw=1.4)
    ax.axhline(1.0, color="gray", lw=0.8, ls="--")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Data")
    ax.set_ylabel("Equity")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
out_img = Path.home() / "Desktop" / "bitcoin" / "stage6_economic_utility.png"
plt.savefig(out_img, dpi=120, bbox_inches="tight")
plt.show()
print(f"\n  💾 Grafico salvato: {out_img.name}")
print("\n✅ STAGE 6 completato.")
